# FreshQA Experiment Runner

This notebook allows you to run experiments using different models and prompt types. You can select a judge model to evaluate the results and generate a report based on the experiments conducted.

In [ ]:
# Colab / environment setup
import os, sys
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print(f"Running in Colab: {IN_COLAB}")

# Optional: install requirements from repository (uncomment to run)
# !pip install -r /content/agentic_search_eval/new_order/colab-freshqa-experiments/requirements.txt

# Optional: mount Google Drive if your data is in Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Ensure the local project src directory is on sys.path so `from src import ...` works
cwd = os.getcwd()
notebooks_dir = os.path.join(cwd, 'new_order', 'colab-freshqa-experiments', 'notebooks')
# try to compute repo root relative to notebooks location; fall back to cwd
if os.path.isdir(notebooks_dir):
    repo_root = os.path.abspath(os.path.join(notebooks_dir, '..'))
else:
    repo_root = cwd
src_path = os.path.abspath(os.path.join(repo_root, 'src'))
project_path = os.path.abspath(os.path.join(repo_root))
print('Computed project_path:', project_path)
print('Computed src_path:', src_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    sys.path.insert(0, project_path)

In [ ]:
# Import necessary libraries and project functions
import os
import json
import pandas as pd
from src.runner import run
from src.judge import run_judge


In [ ]:
# Load the filtered questions (with a small fallback check)
data_path = '../data/freshqa_filtered.csv'
if not os.path.exists(data_path):
    print(f"Default data path not found: {data_path}")
    # common alternate locations in Colab / Drive (adjust as needed)
    alt = '/content/drive/MyDrive/freshqa_filtered.csv'
    if os.path.exists(alt):
        data_path = alt
        print('Found data at:', data_path)
    else:
        print('Data file not found. Please upload data to the notebook environment or mount Drive.')
        # create an empty DataFrame placeholder so later cells don't crash
        import pandas as pd
        questions = pd.DataFrame(columns=['question_id', 'question'])
        print('Using empty questions DataFrame; replace with your CSV to run experiments.')
else:
    questions = pd.read_csv(data_path)
    print(f'Loaded {len(questions)} questions from {data_path}.')

In [ ]:
# Select model, prompt type, and judge model
model_options = ['llama-3-8b']  # Add more models as needed
prompt_variants = ['as_of']  # Add more prompt types as needed
judge_model = 'gemini/gemini-2.5-flash'  # Select judge model

print('Available models:', model_options)
print('Available prompt variants:', prompt_variants)
print('Selected judge model:', judge_model)

In [ ]:
# Run experiments (this uses the lightweight runner that ships in src/)
n_questions = 5  # Number of questions to sample
seed = 42  # Random seed for reproducibility
results_path = 'results.json'  # Path where runner will save results
output_path = 'judgements.json'  # Path to save judgements

for model in model_options:
    for prompt in prompt_variants:
        print(f'Running experiments for model: {model}, prompt: {prompt}')
        # runner.run will create results.json at results_path
        run(n=n_questions, seed=seed, model=model, prompt=prompt, results_path=results_path)

# Run judging on results
if os.path.exists(results_path):
    run_judge(results_path=results_path, output_path=output_path)
    print('Judgements saved to', output_path)
else:
    print('No results file found at', results_path, '— skipping judging step.')

## Conclusion
After running the experiments, the results will be evaluated, and judgements will be saved in the specified output file.